# Tổng hợp so sánh với bài báo

Chỉ nhận đủ ma trận full run; từ chối pilot. Recall và FAR là hai chỉ số phải đọc cùng nhau. Chênh lệch là structural trừ từng đối thủ; trung bình seed trong từng set rồi trọng số set bằng nhau. CI bootstrap mô tả các file đã quan sát, không đại diện quần thể xe.

In [ ]:
from pathlib import Path
import os, subprocess, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
PROJECT_HINT = Path('/content/drive/MyDrive/00_Research')  # sửa nếu đặt dự án nơi khác
candidates = [PROJECT_HINT, Path.cwd(), *Path.cwd().parents]
CODE = next((p / 'code' if (p / 'code/comparison/config.json').exists() else p
             for p in candidates if (p / 'code/comparison/config.json').exists() or
             (p / 'comparison/config.json').exists()), None)
assert CODE is not None, 'Đặt PROJECT_HINT tới thư mục chứa code/'
CODE = CODE.resolve()
CACHE = Path('/content/ctt_compare_cache') if IN_COLAB else CODE / 'results/comparison_data_v1'
OUTPUT = CODE / 'results/comparison_content_v1'
PILOT = CODE / 'results/comparison_pilot'
print('Code:', CODE, '\nCache:', CACHE, '\nResults:', OUTPUT)
def run_stage(stage, output=OUTPUT, extra=()):
    subprocess.run([sys.executable, str(CODE/'scripts/run_comparison.py'), '--stage', stage,
                    '--cache', str(CACHE), '--output', str(output), *map(str, extra)], check=True)


In [ ]:
run_stage('summarize')

In [ ]:
import pandas as pd
SUMMARY = OUTPUT/'summary'
print((SUMMARY/'coverage.json').read_text())
display(pd.read_csv(SUMMARY/'paired_by_set.csv'))
display(pd.read_csv(SUMMARY/'paired_equal_set_mean.csv', header=[0,1]))

In [ ]:
from IPython.display import display, Image
for metric in ['Recall','FAR_per_100k','Attack_F1','AP']:
    print(metric)
    display(Image(filename=str(SUMMARY/(metric+'.png'))))

In [ ]:
intervals = json.loads((SUMMARY/'paired_capture_bootstrap.json').read_text())
display(pd.DataFrame(intervals).query("condition in ['B', 'D']"))

Đọc B làm phép so chính, D để thấy giới hạn. Báo cả A/C, thất bại, class prevalence và mẫu số. Độ lệch chuẩn ba seed không phải khoảng tin cậy cho xe mới. Nếu Recall tăng cùng FAR tăng, mô tả đánh đổi. Cập nhật hai abstract, Chương 5 và kết luận từ số thật sau khi kiểm tra provenance; không lấy số metric công bố trên dataset khác để xếp hạng trực tiếp.